# Lending Club Advanced Interactive Visualization

This notebook loads:
- `artifacts/cleaning_only/cleaned_outputs.joblib`
- `artifacts/cleaning_only/meta.joblib`
- `artifacts/models/trained_models_bundle.joblib`

and provides:
1. Model leaderboard (ROC-AUC / PR-AUC / Brier)
2. Threshold explorer (confusion matrix + score distribution)
3. ROC / PR / calibration dashboard
4. Decile risk + cumulative capture (gains-style)
5. Single-loan score explorer

> Widgets require a live Python kernel (`ipywidgets`) to be interactive.

In [1]:
# Imports
import numpy as np
import pandas as pd
from pathlib import Path
import joblib

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    roc_curve,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

In [2]:
# Load artifacts
CLEANING_DIR = Path("artifacts/cleaning_only")
MODEL_DIR = Path("artifacts/models")

CLEANED_OUTPUTS_PATH = CLEANING_DIR / "cleaned_outputs.joblib"
META_PATH = CLEANING_DIR / "meta.joblib"
TRAINED_BUNDLE_PATH = MODEL_DIR / "trained_models_bundle.joblib"

if not CLEANED_OUTPUTS_PATH.exists():
    raise FileNotFoundError(f"Missing {CLEANED_OUTPUTS_PATH}")
if not META_PATH.exists():
    raise FileNotFoundError(f"Missing {META_PATH}")
if not TRAINED_BUNDLE_PATH.exists():
    raise FileNotFoundError(f"Missing {TRAINED_BUNDLE_PATH}. Run data_modeling.ipynb first.")

cleaned_outputs = joblib.load(CLEANED_OUTPUTS_PATH)
meta = joblib.load(META_PATH)
trained_bundle = joblib.load(TRAINED_BUNDLE_PATH)

print("Loaded cleaned outputs, metadata, and trained bundle.")
print("Meta:", meta)
print("Target note:", meta.get("target_definition_note", "N/A"))
print("Saved thresholds:", trained_bundle.get("thresholds", {}))

Loaded cleaned outputs, metadata, and trained bundle.
Meta: {'snapshot_date': '2018-12-31', 'horizon_months': 12, 'split_years': {'train_end_year': 2015, 'val_year': 2016, 'test_year': 2017}, 'fundamental_drop': ['grade', 'sub_grade', 'int_rate', 'installment'], 'non_default_final_statuses': ['Does not meet the credit policy. Status: Fully Paid', 'Fully Paid'], 'active_nondefault_statuses': ['Current', 'In Grace Period', 'Late (16-30 days)', 'Late (31-120 days)'], 'default_statuses': ['Charged Off', 'Default', 'Does not meet the credit policy. Status: Charged Off'], 'target_definition_note': 'Default timing is approximated using last_pymnt_d because explicit default event timestamps are not included in the selected raw columns.'}
Target note: Default timing is approximated using last_pymnt_d because explicit default event timestamps are not included in the selected raw columns.
Saved thresholds: {'log_fund_val_best_f1_thr': 0.5760515834513242, 'xgb_fund_val_best_f1_thr': 0.589701354503

In [3]:
# Unpack datasets + model specs + safe helpers
X_train_fund_clean = cleaned_outputs["X_train_fund_clean"]
X_val_fund_clean = cleaned_outputs["X_val_fund_clean"]
X_test_fund_clean = cleaned_outputs["X_test_fund_clean"]

X_train_fund_xgb = cleaned_outputs["X_train_fund_xgb"]
X_val_fund_xgb = cleaned_outputs["X_val_fund_xgb"]
X_test_fund_xgb = cleaned_outputs["X_test_fund_xgb"]

X_train_full_xgb = cleaned_outputs["X_train_full_xgb"]
X_val_full_xgb = cleaned_outputs["X_val_full_xgb"]
X_test_full_xgb = cleaned_outputs["X_test_full_xgb"]

y_train_f = np.asarray(cleaned_outputs["y_train_f"]).astype(int)
y_val_f = np.asarray(cleaned_outputs["y_val_f"]).astype(int)
y_test_f = np.asarray(cleaned_outputs["y_test_f"]).astype(int)

y_train_full = np.asarray(cleaned_outputs["y_train_full"]).astype(int)
y_val_full = np.asarray(cleaned_outputs["y_val_full"]).astype(int)
y_test_full = np.asarray(cleaned_outputs["y_test_full"]).astype(int)

# Model specs aligned with data_modeling bundle keys
model_specs = {
    "Logistic (Fundamental)": {
        "X_train": X_train_fund_clean,
        "X_val": X_val_fund_clean,
        "X_test": X_test_fund_clean,
        "y_train": y_train_f,
        "y_val": y_val_f,
        "y_test": y_test_f,
        "bundle_key": "log_fund",
        "encoder_key": "enc_fund_log",
        "threshold_key": "log_fund_val_best_f1_thr",
    },
    "XGB (Fundamental)": {
        "X_train": X_train_fund_xgb,
        "X_val": X_val_fund_xgb,
        "X_test": X_test_fund_xgb,
        "y_train": y_train_f,
        "y_val": y_val_f,
        "y_test": y_test_f,
        "bundle_key": "xgb_fund",
        "encoder_key": "enc_fund_xgb",
        "threshold_key": "xgb_fund_val_best_f1_thr",
    },
    "XGB (Full)": {
        "X_train": X_train_full_xgb,
        "X_val": X_val_full_xgb,
        "X_test": X_test_full_xgb,
        "y_train": y_train_full,
        "y_val": y_val_full,
        "y_test": y_test_full,
        "bundle_key": "xgb_full",
        "encoder_key": "enc_full_xgb",
        "threshold_key": "xgb_full_val_best_f1_thr",
    },
}


def _sanitize_for_encoder(X: pd.DataFrame) -> pd.DataFrame:
    """
    Convert pandas nullable values (pd.NA) to sklearn-friendly values.
    This prevents 'boolean value of NA is ambiguous' errors in encoder.transform.
    """
    X = X.copy()
    for c in X.columns:
        col = X[c]
        if pd.api.types.is_numeric_dtype(col):
            X[c] = pd.to_numeric(col, errors="coerce")
        else:
            col_obj = col.astype("object")
            X[c] = col_obj.where(pd.notna(col_obj), np.nan)
    return X


def _predict_prob(model, encoder, X: pd.DataFrame):
    if encoder is not None:
        X_safe = _sanitize_for_encoder(X)
        X_enc = encoder.transform(X_safe)
    else:
        X_enc = X
    return model.predict_proba(X_enc)[:, 1]


def _calibration_curve_manual(y_true, p, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.digitize(p, bins) - 1
    idx = np.clip(idx, 0, n_bins - 1)
    rows = []
    for b in range(n_bins):
        m = idx == b
        if m.sum() == 0:
            continue
        rows.append({
            "bin": b,
            "count": int(m.sum()),
            "mean_pred": float(np.mean(p[m])),
            "frac_pos": float(np.mean(y_true[m])),
        })
    return pd.DataFrame(rows)

In [4]:
# Build prediction cache
pred_cache = {}
saved_thresholds = trained_bundle.get("thresholds", {})

for model_name, spec in model_specs.items():
    m_key = spec["bundle_key"]
    e_key = spec["encoder_key"]

    if m_key not in trained_bundle:
        print(f"Skipping {model_name}: missing model key '{m_key}'")
        continue

    model = trained_bundle[m_key]
    encoder = trained_bundle.get(e_key, None)

    # sanitize once, reuse for predictions + loan explorer
    X_train_s = _sanitize_for_encoder(spec["X_train"])
    X_val_s = _sanitize_for_encoder(spec["X_val"])
    X_test_s = _sanitize_for_encoder(spec["X_test"])

    try:
        pred_cache[model_name] = {
            "train": _predict_prob(model, encoder, X_train_s),
            "val": _predict_prob(model, encoder, X_val_s),
            "test": _predict_prob(model, encoder, X_test_s),
            "y_train": spec["y_train"],
            "y_val": spec["y_val"],
            "y_test": spec["y_test"],
            "X_train": X_train_s.reset_index(drop=True).copy(),
            "X_val": X_val_s.reset_index(drop=True).copy(),
            "X_test": X_test_s.reset_index(drop=True).copy(),
            "saved_threshold": float(saved_thresholds.get(spec["threshold_key"], 0.50)),
        }
    except Exception as e:
        print(f"Skipping {model_name} due to transform/predict error: {e}")

if len(pred_cache) == 0:
    raise RuntimeError("No model predictions available; check trained_models_bundle.joblib keys.")

print("Available models:", list(pred_cache.keys()))
for k, v in pred_cache.items():
    print(f"{k} saved_threshold={v['saved_threshold']:.4f}")

Available models: ['Logistic (Fundamental)', 'XGB (Fundamental)', 'XGB (Full)']
Logistic (Fundamental) saved_threshold=0.5761
XGB (Fundamental) saved_threshold=0.5897
XGB (Full) saved_threshold=0.6672


In [5]:
# Summary table + leaderboard chart
def compute_metrics(y_true, p):
    return {
        "roc_auc": float(roc_auc_score(y_true, p)),
        "pr_auc": float(average_precision_score(y_true, p)),
        "brier": float(brier_score_loss(y_true, p)),
        "base_rate": float(np.mean(y_true)),
    }

rows = []
for model_name, payload in pred_cache.items():
    for split in ["train", "val", "test"]:
        y = payload[f"y_{split}"]
        p = payload[split]
        rows.append({
            "model": model_name,
            "split": split,
            **compute_metrics(y, p),
        })

summary_long = pd.DataFrame(rows)
display(summary_long.sort_values(["split", "pr_auc"], ascending=[True, False]))

fig_summary = px.bar(
    summary_long[summary_long["split"].isin(["val", "test"])],
    x="model",
    y="pr_auc",
    color="split",
    barmode="group",
    text=summary_long[summary_long["split"].isin(["val", "test"])]["pr_auc"].round(4),
    title="Model Comparison by PR-AUC (Validation/Test)",
    template="plotly_white",
)
fig_summary.update_traces(textposition="outside")
fig_summary.update_layout(yaxis_title="PR-AUC", xaxis_title="Model")
fig_summary.show()

,model,split,roc_auc,pr_auc,brier,base_rate
8,XGB (Full),test,0.712393,0.141002,0.216958,0.061542
5,XGB (Fundamental),test,0.667310,0.110971,0.214419,0.061542
2,Logistic (Fundamental),test,0.650372,0.100494,0.226325,0.061542
6,XGB (Full),train,0.762626,0.167544,0.203140,0.055503
3,XGB (Fundamental),train,0.742304,0.154179,0.210416,0.055503
0,Logistic (Fundamental),train,0.665579,0.100160,0.230208,0.055503
7,XGB (Full),val,0.717834,0.156783,0.221452,0.067794
4,XGB (Fundamental),val,0.672272,0.126729,0.221798,0.067794
1,Logistic (Fundamental),val,0.654211,0.114931,0.233022,0.067794


ROC-AUC (Ranking ability): “How well can the model separate good vs bad borrowers?”

PR-AUC (IMPORTANT for your case): “How good is the model at finding defaults?”

Brier Score (Calibration): “Are predicted probabilities accurate?” Lower = better

The dashboard compares the performance of three models for predicting loan default risk. The XGBoost (Full) model performs best, achieving the highest ROC-AUC (~0.71) and PR-AUC (~0.14), which indicates stronger ability to identify high-risk borrowers compared to the other models. The XGBoost (Fundamental) model also improves over Logistic Regression, showing that non-linear relationships help capture default behavior even without institutional variables.

The difference between validation and test performance is small across all models, which suggest that the models generalize well and are not overfitting. However, the Full model benefits from features like loan grade and interest rate, which already reflect LendingClub’s internal risk assessment, meaning its superior performance comes at the cost of reduced independence.

Overall, the results show that machine learning models can effectively rank borrowers by default risk, and that including richer features significantly improves predictive performance, while simpler models still provide a reasonable baseline.



In [6]:
# Dashboard 1: Threshold explorer
def threshold_dashboard(pred_cache_dict):
    model_w = widgets.Dropdown(
        options=list(pred_cache_dict.keys()),
        value=list(pred_cache_dict.keys())[0],
        description="Model:",
        style={"description_width": "initial"},
    )
    split_w = widgets.RadioButtons(
        options=[("Train", "train"), ("Validation", "val"), ("Test", "test")],
        value="val",
        description="Split:",
        style={"description_width": "initial"},
    )
    threshold_w = widgets.FloatSlider(
        value=pred_cache_dict[model_w.value]["saved_threshold"],
        min=0.01, max=0.99, step=0.01,
        description="Threshold:",
        readout_format=".2f",
        continuous_update=False,
        style={"description_width": "initial"},
    )
    use_saved_thr_btn = widgets.Button(description="Use saved val-F1 threshold", button_style="info")
    normalize_cm_w = widgets.Checkbox(value=False, description="Normalize confusion matrix", indent=False)
    out = widgets.Output()

    def set_saved_threshold(_):
        threshold_w.value = pred_cache_dict[model_w.value]["saved_threshold"]

    def on_model_change(change):
        threshold_w.value = pred_cache_dict[change["new"]]["saved_threshold"]
        render()

    def render(_=None):
        model_name = model_w.value
        split = split_w.value
        thr = threshold_w.value

        y_true = pred_cache_dict[model_name][f"y_{split}"]
        p = pred_cache_dict[model_name][split]
        y_pred = (p >= thr).astype(int)

        cm = confusion_matrix(y_true, y_pred)
        if normalize_cm_w.value:
            cm_plot = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
            cm_text = np.round(cm_plot, 3)
            cm_title = "Confusion Matrix (row-normalized)"
            cm_zmin, cm_zmax = 0.0, 1.0
        else:
            cm_plot = cm
            cm_text = cm
            cm_title = "Confusion Matrix (counts)"
            cm_zmin, cm_zmax = None, None

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=(cm_title, "Predicted Probability Distribution"),
            horizontal_spacing=0.14,
        )

        fig.add_trace(
            go.Heatmap(
                z=cm_plot,
                x=["Pred 0", "Pred 1"],
                y=["True 0", "True 1"],
                text=cm_text,
                texttemplate="%{text}",
                colorscale="Blues",
                zmin=cm_zmin,
                zmax=cm_zmax,
                showscale=False,
            ),
            row=1, col=1
        )

        fig.add_trace(go.Histogram(x=p[y_true == 0], name="True class 0", opacity=0.65, nbinsx=60), row=1, col=2)
        fig.add_trace(go.Histogram(x=p[y_true == 1], name="True class 1", opacity=0.65, nbinsx=60), row=1, col=2)
        fig.add_vline(x=thr, line_dash="dash", line_color="red", row=1, col=2)

        fig.update_layout(
            barmode="overlay",
            template="plotly_white",
            height=700, 
            width=1300,
            title=(
                f"{model_name} | split={split} | threshold={thr:.2f} "
                f"| Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}"
            ),
        )
        fig.update_xaxes(title_text="Predicted probability", row=1, col=2)
        fig.update_yaxes(title_text="Count", row=1, col=2)

        with out:
            out.clear_output(wait=True)
            display(fig)

    model_w.observe(on_model_change, names="value")
    split_w.observe(render, names="value")
    threshold_w.observe(render, names="value")
    normalize_cm_w.observe(render, names="value")
    use_saved_thr_btn.on_click(set_saved_threshold)

    controls = widgets.VBox([model_w, split_w, threshold_w, use_saved_thr_btn, normalize_cm_w])
    display(widgets.HBox([controls, out]))
    render()

threshold_dashboard(pred_cache)

Threshold = decision cutoff

If probability ≥ threshold → classify as default (risky)

Example:

threshold = 0.5 → aggressive detection
threshold = 0.7 → more conservative

Lower threshold: ✅ higher recall (catch more defaults) ❌ lower precision (more false alarms)

Higher threshold: ✅ higher precision ❌ lower recall

The dashboard illustrates how the classification threshold affects model performance for the XGBoost (Full) model. At the selected threshold of approximately 0.67, the model achieves a precision of 0.151, recall of 0.360, and F1-score of 0.212. This means that while the model successfully identifies about 36% of actual defaults, only about 15% of the loans flagged as risky truly default, highlighting the trade-off between detecting defaults and avoiding false positives.

The confusion matrix shows that most loans are correctly classified as non-default (true negatives), but a significant number of non-default loans are still incorrectly flagged as risky (false positives). This is expected due to the class imbalance, where non-default loans dominate the dataset.

The probability distribution plot further shows that predicted probabilities for default and non-default borrowers overlap substantially, which explains why perfect separation is not possible. The threshold line demonstrates how adjusting the cutoff changes classification decisions. Lowering the threshold would increase recall (catch more defaults) but also increase false positives, while raising it would improve precision but miss more risky borrowers.

Overall, this dashboard emphasizes that selecting the right threshold is a business decision: lenders must balance the cost of missed defaults against the cost of incorrectly rejecting good borrowers.

In [7]:
# Dashboard 2: ROC / PR / Calibration
def curves_dashboard(pred_cache_dict):
    split_w = widgets.Dropdown(
        options=[("Train", "train"), ("Validation", "val"), ("Test", "test")],
        value="val",
        description="Split:",
        style={"description_width": "initial"},
    )

    selected_models_w = widgets.SelectMultiple(
        options=list(pred_cache_dict.keys()),
        value=tuple(list(pred_cache_dict.keys())),
        description="Models:",
        rows=3,
        style={"description_width": "initial"},
    )

    bins_w = widgets.IntSlider(
        value=10, min=5, max=30, step=1,
        description="Calibration bins:",
        continuous_update=False,
        style={"description_width": "initial"},
    )

    out = widgets.Output()

    def render(_=None):
        split = split_w.value
        selected = list(selected_models_w.value)
        n_bins = bins_w.value

        with out:
            out.clear_output(wait=True)
            if len(selected) == 0:
                print("Select at least one model.")
                return

            fig = make_subplots(
                rows=1, cols=3,
                subplot_titles=("ROC Curve", "Precision-Recall Curve", "Calibration Curve"),
                horizontal_spacing=0.09,
            )

            for model_name in selected:
                y_true = pred_cache_dict[model_name][f"y_{split}"]
                p = pred_cache_dict[model_name][split]

                fpr, tpr, _ = roc_curve(y_true, p)
                prec, rec, _ = precision_recall_curve(y_true, p)
                calib = _calibration_curve_manual(y_true, p, n_bins=n_bins)

                roc_auc = roc_auc_score(y_true, p)
                pr_auc = average_precision_score(y_true, p)
                brier = brier_score_loss(y_true, p)

                fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name=f"{model_name} (AUC={roc_auc:.3f})", legendgroup=model_name), row=1, col=1)
                fig.add_trace(go.Scatter(x=rec, y=prec, mode="lines", name=f"{model_name} (AP={pr_auc:.3f})", legendgroup=model_name, showlegend=False), row=1, col=2)
                fig.add_trace(
                    go.Scatter(
                        x=calib["mean_pred"], y=calib["frac_pos"],
                        mode="lines+markers",
                        name=f"{model_name} (Brier={brier:.3f})",
                        legendgroup=model_name, showlegend=False
                    ),
                    row=1, col=3
                )

            base_rate = np.mean(pred_cache_dict[selected[0]][f"y_{split}"])
            fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(dash="dash", color="gray"), showlegend=False), row=1, col=1)
            fig.add_trace(go.Scatter(x=[0, 1], y=[base_rate, base_rate], mode="lines", line=dict(dash="dash", color="gray"), showlegend=False), row=1, col=2)
            fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(dash="dash", color="gray"), showlegend=False), row=1, col=3)

            fig.update_xaxes(title_text="FPR", row=1, col=1)
            fig.update_yaxes(title_text="TPR", row=1, col=1)
            fig.update_xaxes(title_text="Recall", row=1, col=2)
            fig.update_yaxes(title_text="Precision", row=1, col=2)
            fig.update_xaxes(title_text="Predicted default probability", row=1, col=3)
            fig.update_yaxes(title_text="Observed default rate", row=1, col=3)

            fig.update_layout(template="plotly_white", height=700, width=1300, title=f"Curves Dashboard | split={split}")
            display(fig)

    split_w.observe(render, names="value")
    selected_models_w.observe(render, names="value")
    bins_w.observe(render, names="value")

    controls = widgets.VBox([split_w, selected_models_w, bins_w])
    display(widgets.HBox([controls, out]))
    render()

curves_dashboard(pred_cache)

👉 More bins: more detail more noise

👉 Fewer bins: smoother less precise

each group:

- compute average prediction
- compute actual default rate

The dashboard evaluates the overall performance and reliability of the three models across different metrics. The ROC curve shows that the XGBoost (Full) model consistently outperforms the others, achieving the highest area under the curve (AUC ≈ 0.718), indicating the strongest ability to distinguish between default and non-default borrowers. The XGBoost (Fundamental) model performs moderately well, while Logistic Regression shows the lowest ranking performance.

The Precision-Recall (PR) curve provides a more realistic view of performance given the class imbalance. All models show relatively low precision at higher recall levels, which is expected since defaults are rare. However, the XGBoost (Full) model maintains better precision across recall levels, confirming its advantage in identifying high-risk borrowers.

The calibration curve highlights how well predicted probabilities match actual default rates. Ideally, predictions should follow the diagonal line (perfect calibration), but all models deviate from this line, indicating that predicted probabilities are not perfectly reliable. The XGBoost (Full) model tends to slightly overestimate risk at higher probability levels, while the other models show similar calibration limitations.

Overall, this dashboard shows that while XGBoost (Full) provides the best ranking performance, all models face challenges in probability calibration and precision due to the imbalanced nature of the dataset.

In [8]:
# Dashboard 3: Decile risk + cumulative capture
def decile_table(y_true, p, n_bins=10):
    d = pd.DataFrame({"y": y_true, "p": p}).copy()
    d["rank"] = d["p"].rank(method="first")
    d["bin"] = pd.qcut(d["rank"], q=n_bins, labels=False) + 1

    agg = (
        d.groupby("bin", as_index=False)
         .agg(n=("y", "size"), defaults=("y", "sum"), mean_score=("p", "mean"))
         .sort_values("bin", ascending=False)
         .reset_index(drop=True)
    )

    total_defaults = max(int(agg["defaults"].sum()), 1)
    agg["default_rate"] = agg["defaults"] / agg["n"]
    agg["cum_defaults"] = agg["defaults"].cumsum()
    agg["cum_capture"] = agg["cum_defaults"] / total_defaults
    agg["population_share"] = agg["n"] / agg["n"].sum()
    agg["cum_population"] = agg["population_share"].cumsum()
    return agg


def decile_dashboard(pred_cache_dict):
    model_w = widgets.Dropdown(
        options=list(pred_cache_dict.keys()),
        value=list(pred_cache_dict.keys())[0],
        description="Model:",
        style={"description_width": "initial"},
    )
    split_w = widgets.Dropdown(
        options=[("Train", "train"), ("Validation", "val"), ("Test", "test")],
        value="test",
        description="Split:",
        style={"description_width": "initial"},
    )
    n_bins_w = widgets.IntSlider(
        value=10, min=5, max=20, step=1,
        description="Risk groups:",
        continuous_update=False,
        style={"description_width": "initial"},
    )

    out = widgets.Output()

    def render(_=None):
        model_name = model_w.value
        split = split_w.value
        n_bins = n_bins_w.value

        y_true = pred_cache_dict[model_name][f"y_{split}"]
        p = pred_cache_dict[model_name][split]
        dt = decile_table(y_true, p, n_bins=n_bins)

        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=("Default Rate by Risk Group", "Cumulative Default Capture"),
            horizontal_spacing=0.16,
        )

        fig.add_trace(
            go.Bar(
                x=dt["bin"].astype(str),
                y=dt["default_rate"],
                text=(dt["default_rate"] * 100).round(2).astype(str) + "%",
                textposition="outside",
                name="Default rate",
            ),
            row=1, col=1
        )

        fig.add_trace(go.Scatter(x=dt["cum_population"], y=dt["cum_capture"], mode="lines+markers", name="Model cumulative capture"), row=1, col=2)
        fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(dash="dash", color="gray"), name="Random baseline"), row=1, col=2)

        fig.update_layout(
            template="plotly_white",
            height=600, 
            width=1100,
            title=f"{model_name} | split={split} | groups={n_bins} (higher group = higher risk)",
            showlegend=True
        )
        fig.update_xaxes(title_text="Risk group (high to low)", row=1, col=1)
        fig.update_yaxes(title_text="Observed default rate", row=1, col=1)
        fig.update_xaxes(title_text="Cumulative population", row=1, col=2, range=[0, 1])
        fig.update_yaxes(title_text="Cumulative defaults captured", row=1, col=2, range=[0, 1])

        with out:
            out.clear_output(wait=True)
            display(fig)
            display(dt)

    model_w.observe(render, names="value")
    split_w.observe(render, names="value")
    n_bins_w.observe(render, names="value")

    controls = widgets.VBox([model_w, split_w, n_bins_w])
    display(widgets.HBox([controls, out]))
    render()

decile_dashboard(pred_cache)

Left chart: Default Rate by Risk Group

You divide borrowers into 10 groups:

- Group 10 = highest risk
- Group 1 = lowest risk

Right chart: Cumulative Capture

👉 This answers:

- “If we only target top X% of borrowers, how many defaults do we catch?”

HUGE insight:

- You can reduce risk by focusing on a small group

The dashboard evaluates how effectively the model ranks borrowers by default risk. Borrowers are divided into ten risk groups (deciles), where Group 10 represents the highest predicted risk and Group 1 the lowest. The left chart shows that the observed default rate decreases steadily from the highest-risk group (~12%) to the lowest-risk group (~1.6%), indicating that the model successfully separates high-risk and low-risk borrowers.

The right chart shows cumulative default capture, which measures how many total defaults are identified as we move through the population from highest to lowest risk. The model captures nearly 20% of all defaults within the top 10% of borrowers and about 60% of defaults within the top 40% of borrowers. This performance is significantly better than the random baseline, demonstrating that the model provides meaningful risk ranking.

Overall, this dashboard shows that the model is highly effective for risk segmentation. In practice, lenders can focus on the highest-risk segments to reduce potential losses, making this analysis especially valuable for prioritizing monitoring, pricing, or approval decisions.


In [9]:
# Dashboard 4: Single-loan score explorer
def loan_explorer_dashboard(pred_cache_dict):
    model_w = widgets.Dropdown(
        options=list(pred_cache_dict.keys()),
        value=list(pred_cache_dict.keys())[0],
        description="Model:",
        style={"description_width": "initial"},
    )
    split_w = widgets.Dropdown(
        options=[("Train", "train"), ("Validation", "val"), ("Test", "test")],
        value="test",
        description="Split:",
        style={"description_width": "initial"},
    )
    idx_w = widgets.IntSlider(
        value=0, min=0, max=10, step=1,
        description="Row index:",
        continuous_update=False,
        style={"description_width": "initial"},
    )
    sort_desc_w = widgets.Checkbox(value=True, description="Sort by predicted risk (desc)", indent=False)
    out = widgets.Output()

    def _get_view(model_name, split):
        X = pred_cache_dict[model_name][f"X_{split}"].copy()
        p = pred_cache_dict[model_name][split]
        y = pred_cache_dict[model_name][f"y_{split}"]

        view = X.reset_index(drop=True).copy()
        view["pred_default_prob"] = p
        view["actual_target"] = y
        return view

    def _refresh_idx_bounds():
        view = _get_view(model_w.value, split_w.value)
        idx_w.max = max(len(view) - 1, 0)
        idx_w.value = min(idx_w.value, idx_w.max)

    def render(_=None):
        view = _get_view(model_w.value, split_w.value)
        if sort_desc_w.value:
            view = view.sort_values("pred_default_prob", ascending=False).reset_index(drop=True)

        i = idx_w.value
        row = view.iloc[i]

        with out:
            out.clear_output(wait=True)

            print(
                f"{model_w.value} | {split_w.value} | row={i} | "
                f"pred_default_prob={row['pred_default_prob']:.4f} | actual_target={int(row['actual_target'])}"
            )

            gauge = go.Figure(go.Indicator(
                mode="gauge+number",
                value=float(row["pred_default_prob"]),
                number={"valueformat": ".2%"},
                title={"text": "Predicted Default Probability"},
                gauge={
                    "axis": {"range": [0, 1]},
                    "bar": {"color": "crimson"},
                    "steps": [
                        {"range": [0.0, 0.2], "color": "#E8F5E9"},
                        {"range": [0.2, 0.5], "color": "#FFF9C4"},
                        {"range": [0.5, 1.0], "color": "#FFEBEE"},
                    ],
                },
            ))
            gauge.update_layout(template="plotly_white", height=280, margin=dict(l=30, r=30, t=45, b=20))
            display(gauge)

            feature_row = row.drop(labels=["pred_default_prob", "actual_target"]).to_frame(name="value")
            feature_row.index.name = "feature"
            display(feature_row)

    def on_model_or_split_change(_):
        _refresh_idx_bounds()
        render()

    model_w.observe(on_model_or_split_change, names="value")
    split_w.observe(on_model_or_split_change, names="value")
    idx_w.observe(render, names="value")
    sort_desc_w.observe(render, names="value")

    _refresh_idx_bounds()
    controls = widgets.VBox([model_w, split_w, idx_w, sort_desc_w])
    display(widgets.HBox([controls, out]))
    render()

loan_explorer_dashboard(pred_cache)

The dashboard provides a detailed view of individual loan predictions, allowing users to examine how the model evaluates a specific borrower. In this example, the XGBoost (Full) model assigns a predicted default probability of approximately 65%, indicating that the borrower is considered moderately high risk. However, the actual outcome shows that this borrower did not default, representing a false positive prediction.

The feature table displays the borrower’s financial and credit profile, including loan characteristics, income, debt-to-income ratio, and credit history. Several factors contribute to the elevated risk score, such as a relatively high interest rate (17.09%), high debt-to-income ratio (34.36), and a lower credit grade (D1). These features are typically associated with higher default risk, which explains the model’s prediction.

This dashboard is valuable for understanding individual model decisions and identifying potential misclassifications. It highlights how the model interprets borrower characteristics and can help analysts assess whether predictions are reasonable or require further refinement. Overall, it demonstrates the practical use of the model in evaluating single-loan risk in real-world scenarios.

## Final Dashboard Conclusion

The interactive dashboard demonstrates how machine learning models can be used to predict and analyze loan default risk. Among the models tested, the XGBoost (Full) model consistently achieves the best performance, with the highest ROC-AUC and PR-AUC scores, indicating strong ability to rank borrowers by risk. However, its superior performance is partly driven by institutional variables such as loan grade and interest rate, which already reflect LendingClub’s internal risk assessment.

The threshold analysis highlights the trade-off between precision and recall, showing that increasing default detection leads to more false positives. It reinforces that threshold selection is a business decision, depending on whether the priority is minimizing missed defaults or avoiding unnecessary rejection of good borrowers.

The ROC and Precision-Recall curves confirm that all models have meaningful predictive power despite the imbalanced dataset, while the calibration analysis shows that predicted probabilities are not perfectly aligned with actual outcomes. It suggests that while the models are effective at ranking risk, further calibration may be needed for accurate probability estimates.

The decile analysis provides strong business insight by demonstrating that the model effectively segments borrowers into risk groups. A significant portion of defaults is concentrated within the highest-risk segments, meaning lenders can focus on a smaller subset of borrowers to reduce potential losses. It highlights the practical value of the model in risk management and decision-making.

Finally, the single-loan explorer illustrates how the model evaluates individual borrowers, showing how specific features contribute to predicted risk. It helps improve interpretability and allows analysts to investigate model decisions and potential misclassifications.

Overall, the dashboard shows that machine learning models can provide meaningful risk ranking, support better lending decisions, and enable targeted risk management strategies, while also emphasizing the importance of threshold selection, feature interpretation, and probability calibration.

In [10]:
# Export static charts
# EXPORT_DIR = Path("artifacts/visualization")
# EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# fig_summary.write_html(EXPORT_DIR / "model_comparison_pr_auc.html", include_plotlyjs="cdn")
# print("Saved:", EXPORT_DIR / "model_comparison_pr_auc.html")